# Main Figure 3

Show how one resource removal affects two small networks. This notebook has no input files. Change the model settings in the next cell to study another network.

## Settings

The two networks differ in their links. Both lose resource 1.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.integrate import solve_ivp

N = 7
DELETED_RESOURCE = 1
INTERACTION_STRENGTH = 1.0
SELF_REGULATION = 0.1
TRANSFER_EFFICIENCY = 0.7
GROWTH_RESOURCE = 1.0
GROWTH_CONSUMER = 0.0
INITIAL_BIOMASS = 0.1
T_FINAL = 10000.0
STORED_TIME_POINTS = 5000
SAVE = True
FIGURE_DIR = Path('figures')

assert N == 7 and DELETED_RESOURCE in (0, 1)
print('Species:', N, '| deleted resource:', DELETED_RESOURCE,
      '| integration:', (0, T_FINAL), '| stored times:', STORED_TIME_POINTS)

## Model

In [ ]:
def make_mechanism_network(second_resource_start):
    links = np.zeros((N, N))
    links[0, 2:-1] = 1
    links[1, second_resource_start:] = 1
    links = INTERACTION_STRENGTH * np.triu(links)
    matrix = links - links.T + SELF_REGULATION * np.eye(N)
    matrix = TRANSFER_EFFICIENCY * np.minimum(matrix, 0) + np.maximum(matrix, 0)
    off_diagonal = matrix.copy()
    np.fill_diagonal(off_diagonal, 0)
    assert np.count_nonzero(off_diagonal, axis=1).min() >= 1
    return matrix, links

growth = np.array([GROWTH_RESOURCE, GROWTH_RESOURCE] +
                  [GROWTH_CONSUMER] * (N - 2))
times = np.linspace(0, T_FINAL, STORED_TIME_POINTS)

def integrate(matrix, initial_state):
    solution = solve_ivp(lambda t, x: x * (growth - matrix @ x),
                         (0, T_FINAL), initial_state, t_eval=times,
                         method='RK45', rtol=1e-6, atol=1e-9)
    if not solution.success or not np.isfinite(solution.y).all():
        raise RuntimeError(solution.message)
    return solution.y

experiments = {}
for name, first_link in [('Highly connected', 3), ('Poorly connected', 4)]:
    matrix, links = make_mechanism_network(first_link)
    baseline = integrate(matrix, np.full(N, INITIAL_BIOMASS))
    baseline_state = baseline[:, -1].copy()
    deleted_state = baseline_state.copy()
    deleted_state[DELETED_RESOURCE] = 0
    after_deletion = integrate(matrix, deleted_state)
    baseline_total = float(baseline_state.sum())
    experiments[name] = dict(links=links, baseline_state=baseline_state,
                             baseline_total=baseline_total,
                             relative_biomass=after_deletion.sum(axis=0) / baseline_total,
                             final_ratio=float(after_deletion[:, -1].sum() / baseline_total))

summary = pd.DataFrame([{'network': name,
                         'directed_links': int(np.count_nonzero(result['links'])),
                         'baseline_biomass': result['baseline_total'],
                         'final_fraction': result['final_ratio']}
                        for name, result in experiments.items()])
print(summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

## Figure

In [ ]:
def draw_network(ax, name, result, label, color):
    positions = {0: (.35, .16), 1: (.65, .16)}
    positions.update({node: (.10 + .20 * (node - 2), .78) for node in range(2, N)})
    for resource, consumer in zip(*np.where(result['links'] > 0)):
        x0, y0 = positions[int(resource)]
        x1, y1 = positions[int(consumer)]
        ax.plot([x0, x1], [y0, y1], color='#bcbcbc', lw=2.5, zorder=1)
    sizes = result['baseline_state'] / max(result['baseline_state'].max(), 1e-12)
    for node, (x, y) in positions.items():
        radius = .038 + .018 * np.sqrt(max(sizes[node], 0))
        face = '#2c91bd' if node < 2 else '#ff8c28'
        ax.add_patch(Circle((x, y), radius, facecolor=face,
                            edgecolor='white', linewidth=1.2, zorder=3))
    x, y = positions[DELETED_RESOURCE]
    ax.plot([x - .075, x + .075], [y - .075, y + .075], color='#ad343a', lw=4, zorder=4)
    ax.plot([x - .075, x + .075], [y + .075, y - .075], color='#ad343a', lw=4, zorder=4)
    ax.text(.02, .95, label, transform=ax.transAxes,
            fontsize=16, fontweight='bold', va='top')
    ax.set_title(name, color=color, fontsize=15)
    ax.set(xlim=(0, 1), ylim=(0, 1))
    ax.set_aspect('equal')
    ax.axis('off')

fig = plt.figure(figsize=(9, 7.5))
grid = fig.add_gridspec(2, 2, height_ratios=[1, 1.35], hspace=.12)
draw_network(fig.add_subplot(grid[0, 0]), 'Highly connected',
             experiments['Highly connected'], 'a', '#ed4548')
draw_network(fig.add_subplot(grid[0, 1]), 'Poorly connected',
             experiments['Poorly connected'], 'b', '#33a9c7')
ax = fig.add_subplot(grid[1, :])
ax.axvspan(-T_FINAL / 3, 0, facecolor='#f8edc6', alpha=.65)
ax.plot([-T_FINAL / 3, 0], [1, 1], color='#666666', lw=2.2)
for name, color in [('Highly connected', '#ed4548'),
                    ('Poorly connected', '#33a9c7')]:
    ax.plot(times, experiments[name]['relative_biomass'], color=color, lw=2.4,
            label=name)
ax.axvline(0, color='#a74747', ls=':', lw=1.8)
ax.text(-.08, 1.03, 'c', transform=ax.transAxes, fontsize=16, fontweight='bold')
ax.set(xlim=(-T_FINAL / 3, T_FINAL), ylim=(.75, 1.1),
       xlabel='Time after species deletion', ylabel='Relative total biomass')
ax.legend(frameon=False)
fig.tight_layout()
if SAVE:
    FIGURE_DIR.mkdir(exist_ok=True)
    for extension in ('png', 'pdf', 'svg'):
        fig.savefig(FIGURE_DIR / f'main_figure_3.{extension}',
                    dpi=300, bbox_inches='tight')
plt.show()